# Classification and Clustering

**Objectives:**
- Preprocess real-world data for machine learning (encoding, variable selection)
- Train classification models (Logistic Regression, K-Nearest Neighbors)
- Evaluate classifiers using accuracy and confusion matrices
- Compare models and validate on held-out data
- Apply K-Means clustering and relate clusters to labels


**How this notebook works:**  
Setup cells (imports, data loading, preprocessing) are already filled in.  
You complete the cells marked with `# TODO`.

## Part 1: Predicting the Credit Score

The two CSV files (source: [Kaggle](https://www.kaggle.com/datasets/clkmuhammed/creditscoreclassification?select=train.csv)) contain data about clients of a global finance company.

The goal is to predict a client's credit score category (**Good / Standard / Poor**) from available features.

We follow a **three-set approach**:
- `train.csv` → split into a **training set** and a **test set** (for model development)
- `test.csv` → kept as a **validation set** (used only at the very end to assess the chosen model)

### Step 1: Import and Explore the Data

In [ ]:
# --- SETUP (run this cell) ---
import pandas
import seaborn as sns
from matplotlib import pyplot as plt
from sklearn import metrics

dataset = pandas.read_csv("train.csv")
validation = pandas.read_csv("test.csv")

In [ ]:
# --- SETUP (run this cell) ---
dataset.columns

### Step 2: Encode the Target Variable

The `Credit_Score` column contains text (`Good`, `Standard`, `Poor`). We convert it to numbers (0, 1, 2) using `LabelEncoder`.

In [ ]:
# --- SETUP (run this cell) ---
dataset['Credit_Score'].unique()

In [ ]:
# --- SETUP (run this cell) ---
from sklearn.preprocessing import LabelEncoder

cle = LabelEncoder()
dataset['Credit_Score'] = cle.fit_transform(dataset['Credit_Score'])

# Check the mapping
score_categories = cle.inverse_transform([0, 1, 2])
print("0 =", score_categories[0], "| 1 =", score_categories[1], "| 2 =", score_categories[2])

### Step 3: Encode Categorical Features & Clean Up

Several features are stored as text. We encode them as numbers and drop the `Name` column.

In [ ]:
# --- SETUP (run this cell) ---
from sklearn.preprocessing import LabelEncoder as le

dataset['Payment_of_Min_Amount'] = le().fit_transform(dataset['Payment_of_Min_Amount'])
dataset['Payment_Behaviour'] = le().fit_transform(dataset['Payment_Behaviour'])
dataset['Occupation'] = le().fit_transform(dataset['Occupation'])
dataset['Type_of_Loan'] = le().fit_transform(dataset['Type_of_Loan'])
dataset['Credit_Mix'] = le().fit_transform(dataset['Credit_Mix'])

dataset = dataset.drop(columns=["Name"], errors='ignore')

### Step 4: Visualize the Data

In [ ]:
# --- SETUP (run this cell) ---
sns.histplot(dataset['Credit_Score'])

In [ ]:
# --- SETUP (run this cell) ---
plt.figure(figsize=(14, 10))
sns.heatmap(dataset.select_dtypes(include='number').corr())
plt.title("Correlation heatmap")
plt.show()

### Step 5: Split Into Training and Test Sets

In [ ]:
# --- SETUP (run this cell) ---
import sklearn.model_selection

df_train, df_test = sklearn.model_selection.train_test_split(
    dataset, test_size=0.25, random_state=243
)

### Step 6: Select Features and Prepare X / Y

We keep only meaningful predictors (no IDs, SSNs, etc.).

In [ ]:
# --- SETUP (run this cell) ---
variables = [
    'Changed_Credit_Limit',
    'Payment_of_Min_Amount',
    'Credit_Mix',
    'Delay_from_due_date',
    'Annual_Income',
    'Monthly_Inhand_Salary',
    'Age',
    'Monthly_Balance',
    'Num_of_Delayed_Payment',
    'Outstanding_Debt',
    'Payment_Behaviour',
    'Credit_History_Age',
    'Num_Bank_Accounts',
    'Credit_Utilization_Ratio'
]

# Training features and labels
X = df_train[variables]
Y = df_train['Credit_Score']

# Test features and labels
X_test = df_test[variables]
Y_test = df_test['Credit_Score']

### Step 7: Logistic Regression

Logistic regression predicts the *probability* of belonging to each category. It is the classification counterpart of linear regression.

Scikit-learn pattern: **create** a model → **fit** it → **score** / **predict**.

In [ ]:
from sklearn.linear_model import LogisticRegression

# TODO: Create a LogisticRegression model (with max_iter=1000) and fit it on X, Y
# Hint: model_lr = LogisticRegression(max_iter=1000)  then  model_lr.fit(...)
# Expected output: LogisticRegression(max_iter=1000)


In [ ]:
# TODO: Print the training accuracy AND test accuracy
# Hint: model_lr.score(X, Y) for training, model_lr.score(X_test, Y_test) for test
# Expected output: two numbers (proportions of correctly classified observations)


### Step 8: Confusion Matrix

**Why?** Accuracy alone can be misleading. The confusion matrix shows us *which* categories are correctly predicted and *where* the model makes mistakes.

- Rows = actual labels
- Columns = predicted labels
- Diagonal = correct predictions

In [ ]:
# TODO: Generate predictions on the test set and compute the confusion matrix
# Hint:
#   actual = Y_test
#   predicted = model_lr.predict(X_test)
#   confusion_matrix = metrics.confusion_matrix(actual, predicted)
# Expected output: a 3x3 numpy array


In [ ]:
# TODO: Display the confusion matrix as a plot
# Hint:
#   cm_display_lr = metrics.ConfusionMatrixDisplay(
#       confusion_matrix=confusion_matrix, display_labels=score_categories
#   )
#   fig, ax = plt.subplots(figsize=(8, 8))
#   ax.grid(False)
#   cm_display_lr.plot(ax=ax)
# Expected output: a labeled heatmap


#### Normalized Confusion Matrices

- **Normalize by row** (divide each row by its sum) → shows **recall**: what fraction of each actual category was correctly detected?
- **Normalize by column** (divide each column by its sum) → shows **precision**: of all predictions for a category, how many were correct?

In [ ]:
# TODO: Compute the row-normalized confusion matrix (recall) and display it
# Hint: cm_by_row = confusion_matrix / confusion_matrix.sum(axis=1)[:, None] * 100
# Then create a ConfusionMatrixDisplay and plot it.
# Expected output: a 3x3 matrix where each row sums to 100


In [ ]:
# TODO: Compute the column-normalized confusion matrix (precision) and display it
# Hint: cm_by_col = confusion_matrix / confusion_matrix.sum(axis=0)[None, :] * 100
# Expected output: a 3x3 matrix where each column sums to 100


> **Question:** What percentage of actual "Poor" clients were correctly detected? What percentage of clients predicted as "Poor" were truly poor?

### Step 9: K-Nearest Neighbors (KNN)

Logistic regression is *linear* — it may miss nonlinear patterns. **KNN** predicts a label based on the majority vote of the $k$ closest training points. It is simple but can capture nonlinear relationships.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# TODO: Create a KNeighborsClassifier, fit it on X, Y
# Hint: model_knn = KNeighborsClassifier()  then  model_knn.fit(X, Y)
# Expected output: KNeighborsClassifier()


In [ ]:
# TODO: Compute predictions and the confusion matrix for KNN on the test set
# Hint: same pattern as logistic regression:
#   predicted = model_knn.predict(X_test)
#   confusion_matrix_knn = metrics.confusion_matrix(Y_test, predicted)
#   cm_display_knn = metrics.ConfusionMatrixDisplay(...)
# Expected output: a 3x3 numpy array


#### Side-by-Side Comparison

In [ ]:
# TODO: Plot both confusion matrices side by side
# Hint:
#   fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 5))
#   cm_display_lr.plot(ax=axes[0]);  axes[0].set_title("Logistic Regression")
#   cm_display_knn.plot(ax=axes[1]); axes[1].set_title("KNN")
#   plt.tight_layout()
# Expected output: two confusion matrix plots next to each other


> **Question:** Which model performs better? Look at the diagonal values. Why might the nonlinear model do better here?

### Step 10: Validate on the Held-Out Set

We chose KNN because it performed better on the test set. To get an unbiased estimate of real-world performance, we evaluate it on the **validation set** (`test.csv`) that was never used during model development.

We must preprocess the validation set the same way as the training data.

In [ ]:
# --- SETUP (run this cell) ---
# Preprocess the validation set identically
from sklearn.preprocessing import LabelEncoder

validation['Payment_of_Min_Amount'] = LabelEncoder().fit_transform(validation['Payment_of_Min_Amount'])
validation['Payment_Behaviour'] = LabelEncoder().fit_transform(validation['Payment_Behaviour'])
validation['Occupation'] = LabelEncoder().fit_transform(validation['Occupation'])
validation['Type_of_Loan'] = LabelEncoder().fit_transform(validation['Type_of_Loan'])
validation['Credit_Mix'] = LabelEncoder().fit_transform(validation['Credit_Mix'])
validation['Credit_Score'] = LabelEncoder().fit_transform(validation['Credit_Score'])

X_valid = validation[variables]
Y_valid = validation['Credit_Score']

In [ ]:
# TODO: Predict on the validation set with KNN and compute the confusion matrix
# Hint:
#   predicted_valid = model_knn.predict(X_valid)
#   cm_valid = metrics.confusion_matrix(Y_valid, predicted_valid)
# Expected output: a 3x3 confusion matrix


In [ ]:
# TODO: Plot two normalized confusion matrices side by side (by column = precision, by row = recall)
# Hint:
#   fig, axes = plt.subplots(1, 2, figsize=(14, 5))
#   cm_col = cm_valid / cm_valid.sum(axis=0)[None, :]
#   metrics.ConfusionMatrixDisplay(cm_col, display_labels=score_categories).plot(ax=axes[0])
#   axes[0].set_title("Precision")
#   cm_row = cm_valid / cm_valid.sum(axis=1)[:, None]
#   metrics.ConfusionMatrixDisplay(cm_row, display_labels=score_categories).plot(ax=axes[1])
#   axes[1].set_title("Recall")
# Expected output: two normalized heatmaps


> **Question:** Are the validation results similar to the test set results? What does that tell you about the model?

## Part 2: Segmenting the Bank Clients (K-Means Clustering)

**Clustering** is *unsupervised* — we do not give the model any labels. It groups observations into clusters based on similarity alone.

**Goal:** Can we find natural groups among clients *without* using the credit score? Do these groups relate to credit quality?

In [ ]:
from sklearn.cluster import KMeans

# TODO: Create a KMeans model with 3 clusters, fit it on numeric columns of the dataset,
#       then assign each client to a cluster.
# Hint:
#   km_model = KMeans(n_clusters=3)
#   km_model.fit(dataset.select_dtypes(include='number'))
#   dataset['cluster'] = km_model.predict(dataset.select_dtypes(include='number'))
# Expected output: the dataset now has a 'cluster' column with values 0, 1, or 2


In [ ]:
# TODO: Check if the clusters are related to the credit score
# Hint: dataset.groupby('cluster')['Credit_Score'].value_counts(normalize=True)
# Expected output: a table showing the proportion of each score category per cluster


> **Question:** Do the proportions of Good / Standard / Poor vary much across clusters? What does this tell us about the relationship between the clusters and creditworthiness?